# Agentic GitHub Benchmark — универсальные модели по API-ключу

Форк `benchmark-task-agentic-github-v2bf642_1_-patched.ipynb`, переписанный под
запуск **вне Kaggle**: вместо `kaggle_secrets` (секреты) и `kaggle_benchmarks`
(`kbench` — реестр моделей `kbench.llms[...]`, движок раундов `llm.prompt(...)`,
декоратор `@kbench.task` и `.evaluate()`) используются:

- обычные переменные окружения / `getpass` для ключей (`GITHUB_TOKEN`,
  `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GOOGLE_API_KEY`);
- собственный лёгкий мульти-провайдерный LLM-клиент (`AnthropicLLMClient`,
  `OpenAILLMClient`, `GoogleLLMClient`) с интерфейсом `.prompt(message, tools=None) -> str`,
  повторяющим интерфейс `kbench.llms[model_id]` — поэтому вся логика раундов,
  бюджета шагов и GitHub-инструментов из оригинала **не меняется**;
- собственный простой раннер вместо `@kbench.task` / `.evaluate()`.

**Что осталось без изменений по сути:** транспортный слой GitHub API, слой
инструментов (`gh_*`, `make_github_tools`, гварды по префиксу ветки), бюджет
шагов (`StepBudget`/`with_budget`), логика раундов (`run_rounds`) и сжатие
журнала вызовов.

**Требования к токену GitHub:** fine-grained PAT c `Contents: RW`,
`Issues: RW`, `Pull requests: RW` (опционально), `Metadata: R`.


In [ ]:
# SDK провайдеров ставим по необходимости — можно закомментировать лишние,
# если используете только часть моделей.
!pip install -q --upgrade requests anthropic openai google-genai


## 0. Ключи API и конфиг репозитория

In [ ]:
import os
from getpass import getpass


def _get_secret(env_name: str, prompt: str, required: bool = True) -> str:
    """Берёт секрет из переменной окружения, иначе спрашивает интерактивно
    (getpass — ввод не отображается и не попадает в историю ноутбука)."""
    value = os.environ.get(env_name)
    if not value:
        value = getpass(f"{prompt} ({env_name}): ")
    if required and not value:
        raise RuntimeError(f"Нужен {env_name}")
    return value


GITHUB_OWNER = "mlaa4ml"
GITHUB_REPO = "KaggleModelsRepo"
GITHUB_BASE_BRANCH = "main"
GITHUB_API = "https://api.github.com"

GITHUB_TOKEN = _get_secret("GITHUB_TOKEN", "GitHub fine-grained PAT")

# Ключи провайдеров — спрашиваются лениво (см. секцию LLM-клиента), но можно
# запросить сразу все три, если планируете гонять все три модели:
ANTHROPIC_API_KEY = _get_secret("ANTHROPIC_API_KEY", "Anthropic API key", required=False)
OPENAI_API_KEY = _get_secret("OPENAI_API_KEY", "OpenAI API key", required=False)
GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY", "Google (Gemini) API key", required=False)

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

REPO_PATH = f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"
MAX_TOOL_OUTPUT = 12000  # символов: длинные ответы режем, чтобы не жечь контекст


## 1. Транспорт: ретраи, rate limit, пагинация

In [ ]:
def _gh_request(method, path, retries=3, **kwargs):
    """Один запрос к GitHub API с ретраями на 5xx и вторичный rate limit."""
    last = None
    for attempt in range(retries):
        resp = requests.request(
            method, f"{GITHUB_API}{path}", headers=GITHUB_HEADERS, timeout=30, **kwargs
        )
        last = resp
        if resp.status_code < 500 and resp.status_code != 403:
            return resp
        if resp.status_code == 403 and "rate limit" not in resp.text.lower():
            return resp  # это отказ по правам, ретраить бессмысленно
        time.sleep(2 ** attempt)
    return last


def _gh_paginate(path, params=None, max_pages=10):
    """Собирает все страницы списочного эндпоинта."""
    params = dict(params or {})
    params.setdefault("per_page", 100)
    items, page = [], 1
    while page <= max_pages:
        params["page"] = page
        resp = _gh_request("GET", path, params=params)
        if resp.status_code != 200:
            raise RuntimeError(f"{resp.status_code}: {resp.text[:300]}")
        batch = resp.json()
        if not batch:
            break
        items.extend(batch)
        if len(batch) < params["per_page"]:
            break
        page += 1
    return items


def _clip(text):
    text = str(text)
    if len(text) <= MAX_TOOL_OUTPUT:
        return text
    return text[:MAX_TOOL_OUTPUT] + f"\n...[обрезано, всего {len(text)} символов]"


def _err(label, resp):
    return f"ERROR: {label} failed ({resp.status_code}): {resp.text[:300]}"

## 2. Расширенный слой GitHub: contents, tree, issues, PR

In [ ]:
def gh_list_branches():
    try:
        return ", ".join(b["name"] for b in _gh_paginate(f"{REPO_PATH}/branches"))
    except RuntimeError as e:
        return f"ERROR: list_branches failed: {e}"


def gh_branch_sha(branch):
    resp = _gh_request("GET", f"{REPO_PATH}/git/ref/heads/{branch}")
    return resp.json()["object"]["sha"] if resp.status_code == 200 else None


def gh_create_branch(branch_name, base=None):
    base = base or GITHUB_BASE_BRANCH
    if gh_branch_sha(branch_name):
        return f"Ветка {branch_name} уже существует."
    base_sha = gh_branch_sha(base)
    if not base_sha:
        return f"ERROR: базовая ветка {base} не найдена"
    resp = _gh_request(
        "POST", f"{REPO_PATH}/git/refs",
        json={"ref": f"refs/heads/{branch_name}", "sha": base_sha},
    )
    if resp.status_code not in (200, 201):
        return _err("create_branch", resp)
    return f"Ветка {branch_name} создана от {base} ({base_sha[:7]})."


def gh_list_files(branch, subdir=""):
    """Рекурсивный листинг файлов ветки (git/trees?recursive=1)."""
    sha = gh_branch_sha(branch)
    if not sha:
        return f"ERROR: ветка {branch} не найдена"
    resp = _gh_request("GET", f"{REPO_PATH}/git/trees/{sha}", params={"recursive": "1"})
    if resp.status_code != 200:
        return _err("list_files", resp)
    rows = [
        f"{n['path']} ({n.get('size', 0)} B)"
        for n in resp.json().get("tree", [])
        if n["type"] == "blob" and n["path"].startswith(subdir)
    ]
    return _clip("\n".join(rows) or "(пусто)")


def gh_read_file(path, branch, start_line=1, max_lines=None):
    resp = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if resp.status_code != 200:
        return f"ERROR: {path} not found on branch {branch} ({resp.status_code})"
    data = resp.json()
    if data.get("encoding") != "base64":
        return f"ERROR: unexpected encoding {data.get('encoding')!r} for {path}"
    text = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
    if start_line > 1 or max_lines:
        lines = text.splitlines()
        end = start_line - 1 + max_lines if max_lines else len(lines)
        text = "\n".join(lines[start_line - 1:end])
    return _clip(text)


def gh_search_code(query, branch, subdir=""):
    """Простой grep по дереву ветки (без Search API — он лагает на свежих коммитах)."""
    listing = gh_list_files(branch, subdir)
    if listing.startswith("ERROR:"):
        return listing
    hits = []
    for row in listing.splitlines():
        path = row.rsplit(" (", 1)[0]
        content = gh_read_file(path, branch)
        if content.startswith("ERROR:"):
            continue
        for i, line in enumerate(content.splitlines(), start=1):
            if query.lower() in line.lower():
                hits.append(f"{path}:{i}: {line.strip()[:200]}")
                if len(hits) >= 100:
                    return _clip("\n".join(hits))
    return _clip("\n".join(hits) or f"Ничего не найдено по '{query}'")


def gh_write_file(path, content, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    sha = existing.json().get("sha") if existing.status_code == 200 else None
    payload = {
        "message": message or f"{path}: {'update' if sha else 'create'} ({branch})",
        "content": base64.b64encode(content.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }
    if sha:
        payload["sha"] = sha
    resp = _gh_request("PUT", f"{REPO_PATH}/contents/{path}", json=payload)
    if resp.status_code not in (200, 201):
        return _err("write_file", resp)
    commit = resp.json().get("commit", {}).get("sha", "")[:7]
    return f"Записано {len(content)} символов в {path} (ветка {branch}), commit {commit}"


def gh_delete_file(path, branch, message=None):
    existing = _gh_request("GET", f"{REPO_PATH}/contents/{path}", params={"ref": branch})
    if existing.status_code != 200:
        return f"ERROR: {path} нет в ветке {branch}"
    resp = _gh_request("DELETE", f"{REPO_PATH}/contents/{path}", json={
        "message": message or f"{path}: delete ({branch})",
        "sha": existing.json()["sha"],
        "branch": branch,
    })
    return f"Удалён {path} из {branch}" if resp.status_code == 200 else _err("delete_file", resp)


def gh_compare(base, head):
    resp = _gh_request("GET", f"{REPO_PATH}/compare/{base}...{head}")
    if resp.status_code != 200:
        return None
    d = resp.json()
    return {
        "ahead_by": d.get("ahead_by", 0),
        "behind_by": d.get("behind_by", 0),
        "files": [f["filename"] for f in d.get("files", [])],
        "commits": [c["commit"]["message"].splitlines()[0] for c in d.get("commits", [])],
    }


# ---------------- Issues ----------------

def gh_list_issues(state="all", labels=None):
    """Все issues репозитория. PR отфильтрованы (GitHub отдаёт их тем же эндпоинтом)."""
    params = {"state": state}
    if labels:
        params["labels"] = labels
    try:
        items = _gh_paginate(f"{REPO_PATH}/issues", params=params)
    except RuntimeError as e:
        return f"ERROR: list_issues failed: {e}"
    rows = []
    for it in items:
        if "pull_request" in it:
            continue
        lbl = ",".join(l["name"] for l in it.get("labels", [])) or "-"
        rows.append(
            f"#{it['number']} [{it['state']}] {it['title']} "
            f"(labels: {lbl}; comments: {it.get('comments', 0)})"
        )
    return _clip("\n".join(rows) or "Открытых/закрытых issues нет.")


def gh_get_issue(number):
    resp = _gh_request("GET", f"{REPO_PATH}/issues/{number}")
    if resp.status_code != 200:
        return _err(f"get_issue #{number}", resp)
    it = resp.json()
    out = [
        f"ISSUE #{it['number']}: {it['title']}",
        f"state: {it['state']} | author: {it['user']['login']} | "
        f"labels: {','.join(l['name'] for l in it.get('labels', [])) or '-'}",
        f"url: {it['html_url']}",
        "",
        "--- BODY ---",
        it.get("body") or "(пусто)",
    ]
    try:
        for c in _gh_paginate(f"{REPO_PATH}/issues/{number}/comments"):
            out += ["", f"--- COMMENT by {c['user']['login']} ---", c.get("body") or ""]
    except RuntimeError:
        out.append("\n(комментарии недоступны)")
    return _clip("\n".join(out))


def gh_comment_issue(number, body):
    resp = _gh_request("POST", f"{REPO_PATH}/issues/{number}/comments", json={"body": body})
    if resp.status_code != 201:
        return _err(f"comment_issue #{number}", resp) + "  (нужен scope Issues: RW)"
    return f"Комментарий опубликован: {resp.json()['html_url']}"


def gh_create_pull_request(head, base, title, body):
    resp = _gh_request("POST", f"{REPO_PATH}/pulls", json={
        "head": head, "base": base, "title": title, "body": body,
    })
    if resp.status_code != 201:
        return _err("create_pull_request", resp) + "  (нужен scope Pull requests: RW)"
    return f"PR создан: {resp.json()['html_url']}"

## 2b. Патчи issue #16 (`build_toolset`, слой поверх `gh_*` из раздела 2)

Вставлено из `benchmark-task-agentic-github-claude-opus-5-issue16.ipynb` — после раздела 2 (`gh_*`) и до регистрации инструментов (раздел 4). Даёт отдельную точку входа `build_toolset(slug) -> (ctx, tools)` с состоянием рабочей ветки на диске, чтением по умолчанию из рабочей ветки, курсорным чтением больших/минифицированных файлов и исполнением кода. Функции-инструменты из этого слоя нужно регистрировать через `functools.wraps` (см. `with_budget` в разделе 3) — иначе схема инструмента вырождается в `{args, kwargs}` (известный баг проекта).

# Патчи к `benchmark-task-agentic-github-v2bf642(1).ipynb` — issue #16

Автор: агент `anthropic/claude-opus-5@default`, ветка `anthropic-claude-opus-5-default-issue-16`.
Оригинальные ноутбуки не изменены — это отдельный файл с ячейками-заменами.

Ячейки ниже добавляются **после** ячейки со слоем `gh_*` (раздел 2 исходного
ноутбука) и **до** регистрации инструментов агента. Они закрывают 4 пункта issue:

| # | Проблема | Решение в этом ноутбуке |
|---|---|---|
| 1 | Рабочая ветка сбрасывается между раундами (лишний коммит `911c83b`) | `AgentContext` с состоянием на диске (`kbench_agent_state.json`) + guard по slug |
| 2 | `list_files/search_code/read_file` читают `main`, а не рабочую ветку | `build_read_tools(ctx)` — дефолт `from_my_branch=True`, в ответе печатается имя ветки |
| 3 | `MAX_TOOL_OUTPUT` режет ~50 КБ ноутбук | `gh_read_cursor` (курсор `next_start_line`) + `nb_outline` / `nb_cell` / `nb_replace_cell` |
| 4 | Нет исполнения кода и создания issues | `run_python`, `run_repo_file`, `create_issue` |

Требования к токену прежние: `Contents: RW`, `Issues: RW`, `Metadata: R`.

## Fix 1. Рабочая ветка живёт между раундами

In [ ]:
# Проблема: выбранная через create_issue_branch(N) ветка хранилась в переменной
# одного прогона. В новом раунде объект пересоздавался, и первый write_file уходил
# в дефолтную ветку slug (лишний коммит 911c83b в anthropic-claude-opus-5-default).
# Решение: контекст агента с состоянием на диске + жёсткий guard по префиксу slug.
import json
import os

AGENT_STATE_PATH = os.environ.get('KBENCH_STATE', './kbench_agent_state.json')


def _state_load():
    try:
        with open(AGENT_STATE_PATH, encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return {}


def _state_save(state):
    try:
        with open(AGENT_STATE_PATH, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False)
    except Exception as e:
        print('state save failed:', e)


class AgentContext:
    '''Контекст одного агента: slug модели + текущая рабочая ветка.

    Ветка пишется и в память, и на диск, поэтому переживает границу раунда
    и рестарт процесса. Записать можно только в ветку с префиксом slug.
    '''

    def __init__(self, slug, base=None):
        self.slug = slug
        self.base = base or GITHUB_BASE_BRANCH
        self.work_branch = _state_load().get(slug, {}).get('work_branch') or slug

    def set_branch(self, branch):
        if not branch.startswith(self.slug):
            raise ValueError('guard: ветка ' + branch + ' вне slug ' + self.slug)
        self.work_branch = branch
        st = _state_load()
        st[self.slug] = {'work_branch': branch}
        _state_save(st)
        return branch

    def resolve(self, from_my_branch=True, branch=None):
        '''Единая точка выбора ветки для всех инструментов.'''
        if branch:
            return branch
        return self.work_branch if from_my_branch else self.base

    def round_header(self, round_no, rounds_total):
        '''Строка, которую раннер обязан добавлять в промпт каждого раунда.'''
        return ('Раунд ' + str(round_no) + '/' + str(rounds_total) +
                '. Текущая рабочая ветка: ' + self.work_branch +
                ' (все записи идут туда, main read-only).')


def make_branch_tools(ctx):
    def create_issue_branch(issue_number: int) -> str:
        '''Создаёт (идемпотентно) ветку <slug>-issue-<N> и делает её текущей на все раунды.'''
        name = ctx.slug + '-issue-' + str(int(issue_number))
        msg = gh_create_branch(name, ctx.base)
        ctx.set_branch(name)
        return msg + ' Текущая рабочая ветка: ' + name

    def use_branch(branch: str) -> str:
        '''Явно переключить рабочую ветку (только внутри своего slug).'''
        try:
            ctx.set_branch(branch)
        except ValueError as e:
            return 'ERROR: ' + str(e)
        return 'Рабочая ветка: ' + ctx.work_branch

    def current_branch() -> str:
        '''Показать текущую рабочую ветку (дешёвая проверка в начале раунда).'''
        return ctx.work_branch

    return create_issue_branch, use_branch, current_branch

## Fix 2. Чтение по умолчанию из рабочей ветки

In [ ]:
# Проблема: list_files/search_code/read_file молча читали main, поэтому агент
# не видел собственных коммитов и мог перезаписать свою же работу.
# Решение: дефолт from_my_branch=True + имя ветки в первой строке ответа.

def build_read_tools(ctx):
    def list_files(subdir: str = '', from_my_branch: bool = True) -> str:
        '''Листинг файлов. По умолчанию — текущая рабочая ветка агента.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_list_files(br, subdir)

    def search_code(query: str, subdir: str = '', from_my_branch: bool = True) -> str:
        '''grep по дереву ветки (по умолчанию — рабочей). Возвращает path:line: текст.'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_search_code(query, br, subdir)

    def read_file(path: str, start_line: int = 1, max_lines: int = 0,
                  from_my_branch: bool = True) -> str:
        '''Чтение файла из рабочей ветки; from_my_branch=False — из base (main).'''
        br = ctx.resolve(from_my_branch)
        return '[branch: ' + br + ']\n' + gh_read_cursor(path, br, start_line, max_lines or None)

    def diff_vs_base() -> str:
        '''Что рабочая ветка меняет относительно base.'''
        d = gh_compare(ctx.base, ctx.work_branch)
        if not d:
            return 'ERROR: сравнение недоступно'
        return ('ahead_by=' + str(d['ahead_by']) + ', behind_by=' + str(d['behind_by']) +
                '\nfiles: ' + ', '.join(d['files']) +
                '\ncommits: ' + ' | '.join(d['commits']))

    return list_files, search_code, read_file, diff_vs_base

## Fix 3. Большие файлы и ноутбуки: курсор вместо слепой обрезки

In [ ]:
# Проблема: _clip резал ответ на MAX_TOOL_OUTPUT без указания, где остановился;
# минифицированный ipynb (одна строка на ~49 КБ) нельзя было прочитать вообще.
# Решение: страничное чтение с курсором + инструменты, понимающие структуру ipynb.

def _gh_text(path, branch):
    '''Сырой текст файла ветки, без обрезки.'''
    resp = _gh_request('GET', REPO_PATH + '/contents/' + path, params={'ref': branch})
    if resp.status_code != 200:
        return None
    data = resp.json()
    if data.get('encoding') != 'base64':
        return None
    return base64.b64decode(data['content']).decode('utf-8', errors='replace')


def gh_read_cursor(path, branch, start_line=1, max_lines=None):
    '''Читает срез файла и всегда сообщает, где остановился.

    Хвост ответа: ...[cursor] next_start_line=N из M — агент продолжает чтение
    следующим вызовом, а не теряет остаток файла.
    Отдельно обрабатывается случай одной сверхдлинной строки (минифицированный JSON):
    она отдаётся кусками по символам.
    '''
    text = _gh_text(path, branch)
    if text is None:
        return 'ERROR: ' + path + ' недоступен в ветке ' + branch
    lines = text.splitlines()
    total = len(lines)
    if total == 1 and len(text) > MAX_TOOL_OUTPUT:
        off = max(0, int(start_line) - 1)
        piece = text[off:off + MAX_TOOL_OUTPUT - 200]
        nxt = off + len(piece) + 1
        tail = ('\n...[cursor] one-line file, next_start_line=' + str(nxt) +
                ' (позиция в символах) из ' + str(len(text)))
        return piece + (tail if nxt <= len(text) else '\n...[eof]')
    budget = MAX_TOOL_OUTPUT - 200
    start = max(1, int(start_line))
    limit = total if not max_lines else min(total, start - 1 + int(max_lines))
    out, used, i = [], 0, start - 1
    while i < limit and used + len(lines[i]) + 1 <= budget:
        out.append(lines[i])
        used += len(lines[i]) + 1
        i += 1
    if i == start - 1 and i < limit:
        out.append(lines[i][:budget])
        i += 1
    if i < total:
        tail = '\n...[cursor] next_start_line=' + str(i + 1) + ' из ' + str(total)
    else:
        tail = '\n...[eof] всего строк ' + str(total)
    return '\n'.join(out) + tail


def _cell_src(cell):
    src = cell.get('source')
    return ''.join(src) if isinstance(src, list) else (src or '')


def build_nb_tools(ctx):
    def nb_outline(path: str, from_my_branch: bool = True) -> str:
        '''Оглавление ipynb: индекс, тип, размер и первая строка каждой ячейки.

        Дешёвая карта 50 КБ ноутбука вместо чтения его целиком.
        '''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        try:
            nb = json.loads(text)
        except ValueError as e:
            return 'ERROR: не JSON: ' + str(e)
        rows = []
        for i, c in enumerate(nb.get('cells', [])):
            s = _cell_src(c)
            head = (s.strip().splitlines() or [''])[0][:100]
            rows.append(str(i) + ' [' + c.get('cell_type', '?') + '] ' +
                        str(len(s)) + ' B | ' + head)
        return _clip('cells: ' + str(len(rows)) + '\n' + '\n'.join(rows))

    def nb_cell(path: str, index: int, from_my_branch: bool = True) -> str:
        '''Исходник одной ячейки ipynb по индексу.'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        cells = json.loads(text).get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        return _clip(_cell_src(cells[i]))

    def nb_replace_cell(path: str, index: int, new_source: str, message: str = '') -> str:
        '''Точечно заменяет исходник одной ячейки: не надо перезаписывать весь JSON.'''
        br = ctx.work_branch
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        nb = json.loads(text)
        cells = nb.get('cells', [])
        i = int(index)
        if i < 0 or i >= len(cells):
            return 'ERROR: индекс вне диапазона 0..' + str(len(cells) - 1)
        cells[i]['source'] = new_source
        if cells[i].get('cell_type') == 'code':
            cells[i]['outputs'] = []
            cells[i]['execution_count'] = None
        dumped = json.dumps(nb, ensure_ascii=False, indent=1)
        return gh_write_file(path, dumped, br,
                            message or (path + ': заменена ячейка ' + str(i)))

    return nb_outline, nb_cell, nb_replace_cell

## Fix 4. Исполнение кода и создание issues

In [ ]:
# Проблема: агент не мог ни запустить свой код (писал вслепую), ни завести issue.
# Решение: изолированный запуск в отдельном процессе с таймаутом + POST /issues.
import subprocess
import sys
import tempfile
import textwrap

RUN_TIMEOUT_DEFAULT = 30


def _run_file(dirpath, filename, argv, timeout_sec):
    try:
        p = subprocess.run(
            [sys.executable, filename] + argv,
            capture_output=True, text=True, timeout=timeout_sec, cwd=dirpath,
            env={'PATH': os.environ.get('PATH', ''), 'HOME': dirpath,
                 'PYTHONDONTWRITEBYTECODE': '1'},
        )
    except subprocess.TimeoutExpired:
        return 'ERROR: превышен таймаут ' + str(timeout_sec) + ' c'
    return _clip('exit=' + str(p.returncode) +
                 '\n--- stdout ---\n' + p.stdout +
                 '\n--- stderr ---\n' + p.stderr)


def run_python(code: str, timeout_sec: int = RUN_TIMEOUT_DEFAULT) -> str:
    '''Выполняет Python-код в отдельном процессе и возвращает exit code, stdout, stderr.

    Окружение урезано (нет GITHUB_TOKEN в env), рабочая директория временная,
    есть таймаут — чтобы агент проверял свои правки, а не гадал.
    '''
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, 'snippet.py'), 'w', encoding='utf-8') as fh:
            fh.write(textwrap.dedent(code))
        return _run_file(d, 'snippet.py', [], timeout_sec)


def make_run_repo_file(ctx):
    def run_repo_file(path: str, args: str = '', timeout_sec: int = 60,
                      from_my_branch: bool = True) -> str:
        '''Скачивает файл из ветки во временный каталог и запускает его (например, тесты).'''
        br = ctx.resolve(from_my_branch)
        text = _gh_text(path, br)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + br
        with tempfile.TemporaryDirectory() as d:
            name = os.path.basename(path)
            with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
                fh.write(text)
            return _run_file(d, name, args.split() if args else [], timeout_sec)

    return run_repo_file


def create_issue(title: str, body: str = '', labels: str = '') -> str:
    '''Создаёт новый issue в репозитории (нужен scope Issues: Read and write).'''
    payload = {'title': title, 'body': body}
    if labels:
        payload['labels'] = [x.strip() for x in labels.split(',') if x.strip()]
    resp = _gh_request('POST', REPO_PATH + '/issues', json=payload)
    if resp.status_code != 201:
        return _err('create_issue', resp) + '  (нужен scope Issues: RW)'
    return 'Issue создан: ' + resp.json()['html_url']

## Сборка набора инструментов и что поменять в раннере

In [ ]:
import functools


def build_toolset(slug, base=None):
    '''Возвращает (ctx, tools) для одной модели. ctx создаётся ОДИН раз и
    переиспользуется во всех раундах — именно это чинит пункт 1 issue #16.'''
    ctx = AgentContext(slug, base)
    create_issue_branch, use_branch, current_branch = make_branch_tools(ctx)
    list_files, search_code, read_file, diff_vs_base = build_read_tools(ctx)
    nb_outline, nb_cell, nb_replace_cell = build_nb_tools(ctx)
    run_repo_file = make_run_repo_file(ctx)

    def write_file(path: str, content: str, commit_message: str = '') -> str:
        '''Полная запись файла в текущую рабочую ветку.'''
        return gh_write_file(path, content, ctx.work_branch, commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        '''Замена первого вхождения old_text на new_text в файле рабочей ветки.'''
        text = _gh_text(path, ctx.work_branch)
        if text is None:
            return 'ERROR: ' + path + ' недоступен в ветке ' + ctx.work_branch
        if old_text not in text:
            return 'ERROR: фрагмент не найден'
        return gh_write_file(path, text.replace(old_text, new_text, 1), ctx.work_branch,
                             path + ': patch')

    tools = [
        current_branch, create_issue_branch, use_branch,
        list_files, search_code, read_file, diff_vs_base,
        nb_outline, nb_cell, nb_replace_cell,
        write_file, patch_file,
        run_python, run_repo_file, create_issue,
    ]
    return ctx, tools


# Чек-лист изменений в раннере (движок раундов):
# 1. ctx/tools строятся один раз на модель, а не на раунд;
# 2. в системный промпт каждого раунда добавляется ctx.round_header(i, n)
#    -> модель видит текущую ветку и не пишет в дефолтную;
# 3. состояние дублируется в kbench_agent_state.json (переживает рестарт ядра);
# 4. функции-инструменты регистрируются с functools.wraps, иначе схема
#    вырождается в {args, kwargs} (известный баг проекта);
# 5. критерий github_issue_resolution дополняется проверкой,
#    что коммиты попали именно в <slug>-issue-<N>, а не в <slug>.

## 3. Бюджет шагов (без изменений, `functools.wraps` обязателен)

In [ ]:
class StepBudget:
    def __init__(self, limit, log=None):
        self.limit, self.used = limit, 0
        # log — общий список на ВСЮ задачу (все раунды), а не только на этот
        # раунд: сюда пишет with_budget при каждом реальном вызове инструмента.
        # Это и есть память между раундами — объективная, не зависящая от
        # того, напишет ли модель честный текстовый самоотчёт.
        self.log = log if log is not None else []

    def consume(self):
        if self.used >= self.limit:
            return False
        self.used += 1
        return True

    @property
    def remaining(self):
        return max(0, self.limit - self.used)


def with_budget(tool_fn, budget):
    """functools.wraps обязателен: иначе kbench не построит схему параметров."""
    @functools.wraps(tool_fn)
    def wrapped(*args, **kwargs):
        if not budget.consume():
            return (
                f"ERROR: бюджет шагов на раунд исчерпан ({budget.limit}). "
                "Заверши раунд текстовым ответом."
            )
        try:
            result = tool_fn(*args, **kwargs)
        except Exception as e:  # noqa: BLE001 — агент должен видеть текст, а не падение
            result = f"ERROR: {type(e).__name__}: {e}"
        # фиксируем реальный вызов в общий лог задачи (переживает смену раунда)
        budget.log.append({
            "tool": tool_fn.__name__,
            "args": args,
            "kwargs": kwargs,
            "result_preview": str(result)[:20000],
        })
        return result
    return wrapped


# Наш собственный LLM-клиент (см. следующую секцию) сам ограничивает один
# вызов .prompt(...) TOOL_ROUND_LIMIT "раундами" инструментов и кидает
# ToolRoundLimitExceeded при превышении — держим бюджет заметно ниже этого
# внутреннего лимита (тот же принцип, что был у kaggle_benchmarks).
STEPS_PER_ROUND = 10
MAX_ROUNDS = 3


## 4. Расширенный набор инструментов агента

Guard: агент может писать **только** в ветки с префиксом собственного slug.
Имя ветки нигде не приходит от модели напрямую — оно вычисляется из номера issue.

In [ ]:
def slugify_model_id(model_id):
    return re.sub(r"[^a-z0-9]+", "-", model_id.lower()).strip("-")


def make_github_tools(slug, budget, allow_issue_write=True):
    """Возвращает список инструментов, замкнутых на slug модели."""
    state = {"branch": slug}  # текущая рабочая ветка агента

    def _guard(branch):
        if branch != slug and not branch.startswith(slug + "-issue-"):
            raise PermissionError(f"запись в ветку {branch} запрещена (не твой префикс)")
        return branch

    def list_branches() -> str:
        """List all branch names in the repository."""
        return gh_list_branches()

    def create_branch() -> str:
        """Create your main assigned branch (idempotent)."""
        state["branch"] = slug
        return gh_create_branch(slug)

    def create_issue_branch(issue_number: int) -> str:
        """Create and select a dedicated branch for an issue: <your-slug>-issue-<N>."""
        name = _guard(f"{slug}-issue-{int(issue_number)}")
        result = gh_create_branch(name)
        if result.startswith("ERROR:"):
            return result  # рабочую ветку не переключаем — не маскируем неудачное создание
        state["branch"] = name
        return result + f" Текущая рабочая ветка: {name}"

    def list_files(subdir: str = "") -> str:
        """List all files in the base branch, optionally filtered by path prefix."""
        return gh_list_files(GITHUB_BASE_BRANCH, subdir)

    def read_file(path: str, from_my_branch: bool = False,
                  start_line: int = 1, max_lines: int = 0) -> str:
        """Read a file. from_my_branch=True reads your working branch instead of base.
        Use start_line/max_lines to read only a slice of a large file."""
        branch = state["branch"] if from_my_branch else GITHUB_BASE_BRANCH
        return gh_read_file(path, branch, start_line, max_lines or None)

    def search_code(query: str, subdir: str = "") -> str:
        """Grep the base branch for a substring. Returns 'path:line: text' matches."""
        return gh_search_code(query, GITHUB_BASE_BRANCH, subdir)

    def write_file(path: str, content: str, commit_message: str = "") -> str:
        """Create or overwrite a file with FULL content on your working branch."""
        return gh_write_file(path, content, _guard(state["branch"]), commit_message or None)

    def patch_file(path: str, old_text: str, new_text: str) -> str:
        """Replace the first occurrence of old_text with new_text in a file
        on your working branch. Cheaper and safer than rewriting a whole file."""
        branch = _guard(state["branch"])
        current = gh_read_file(path, branch)
        if current.startswith("ERROR:"):
            return current
        if old_text not in current:
            return "ERROR: old_text не найден — прочитай файл и повтори точную подстроку"
        updated = current.replace(old_text, new_text, 1)
        return gh_write_file(path, updated, branch, f"{path}: patch")

    def delete_file(path: str) -> str:
        """Delete a file on your working branch."""
        return gh_delete_file(path, _guard(state["branch"]))

    def diff_vs_base() -> str:
        """Show how your working branch differs from the base branch."""
        cmp = gh_compare(GITHUB_BASE_BRANCH, state["branch"])
        if cmp is None:
            return f"ERROR: не удалось сравнить {state['branch']} с {GITHUB_BASE_BRANCH}"
        return (
            f"branch={state['branch']} ahead_by={cmp['ahead_by']} behind_by={cmp['behind_by']}\n"
            f"files: {', '.join(cmp['files']) or '-'}\n"
            f"commits: {'; '.join(cmp['commits']) or '-'}"
        )

    def list_issues(state_filter: str = "all", labels: str = "") -> str:
        """List ALL issues (open and closed) with number, state, title and labels."""
        return gh_list_issues(state_filter, labels or None)

    def get_issue(issue_number: int) -> str:
        """Read the full body and all comments of one issue."""
        return gh_get_issue(int(issue_number))

    def comment_issue(issue_number: int, body: str) -> str:
        """Post a comment on an issue."""
        return gh_comment_issue(int(issue_number), body)

    def report_work(issue_number: int, summary: str) -> str:
        """Post a structured report on the issue: link to your branch, diff link,
        changed files and your summary. Use this to close the loop on a task."""
        branch = state["branch"]
        cmp = gh_compare(GITHUB_BASE_BRANCH, branch) or {"files": [], "commits": [], "ahead_by": 0}
        body = (
            f"### Работа по issue #{int(issue_number)}\n\n"
            f"{summary}\n\n"
            f"**Ветка:** [`{branch}`]({REPO_URL}/tree/{branch})\n"
            f"**Diff:** {REPO_URL}/compare/{GITHUB_BASE_BRANCH}...{branch}\n"
            f"**Коммитов впереди `{GITHUB_BASE_BRANCH}`:** {cmp['ahead_by']}\n"
            f"**Изменённые файлы:** "
            + (", ".join(f"`{f}`" for f in cmp["files"]) or "—")
            + "\n\n<sub>Автоматический отчёт агента.</sub>"
        )
        return gh_comment_issue(int(issue_number), body)

    def open_pull_request(title: str, body: str = "") -> str:
        """Open a pull request from your working branch into the base branch."""
        return gh_create_pull_request(state["branch"], GITHUB_BASE_BRANCH, title, body)

    tools = [
        list_branches, create_branch, create_issue_branch,
        list_files, read_file, search_code,
        write_file, patch_file, delete_file, diff_vs_base,
    ]
    if allow_issue_write:
        tools += [list_issues, get_issue, comment_issue, report_work, open_pull_request]
    else:
        tools += [list_issues, get_issue]
    return [with_budget(t, budget) for t in tools]

## Универсальный LLM-клиент по API-ключу (замена `kbench.llms[...]`)

Даёт объект с интерфейсом `.prompt(message, tools=None) -> str`, идентичным
`kbench.llms[model_id]` из оригинала — поэтому `StepBudget`, `with_budget`,
`make_github_tools` и (ниже) `run_rounds` работают без изменений.

- Схема инструмента строится из сигнатуры и докстринга python-функции
  (`inspect.signature` + первая строка докстринга) — как раньше это делал
  `kbench` через `functools.wraps`.
- Внутри одного `.prompt(...)` клиент сам крутит цикл tool-calling до
  `TOOL_ROUND_LIMIT` раз; при превышении кидает `ToolRoundLimitExceeded`
  (аналог `ToolInvocationLimitExhausted` у kbench) — эту ошибку уже ловит
  `run_rounds` в секции 6 оригинала.
- Токены/число вызовов накапливаются в `self.usage` для отчёта по стоимости.


In [ ]:
import inspect
import json
import typing

VERBOSE = True  # печатать имена вызываемых инструментов (замена console mode kbench)

# Тот же принцип, что раньше держал kaggle_benchmarks.tools.native: один
# вызов .prompt(...) не может звать инструменты бесконечно.
TOOL_ROUND_LIMIT = 10

_PY_TYPE_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean"}


class ToolRoundLimitExceeded(RuntimeError):
    """Аналог kbench.ToolInvocationLimitExhausted."""


def _tool_schema(fn):
    """JSON Schema инструмента из сигнатуры и докстринга — провайдеро-независимый
    базовый вид; каждый клиент адаптирует его под свой формат tools=..."""
    sig = inspect.signature(fn)
    try:
        hints = typing.get_type_hints(fn)
    except Exception:  # noqa: BLE001 — на всякий случай не роняем схему из-за аннотаций
        hints = {}
    props, required = {}, []
    for name, param in sig.parameters.items():
        if name == "self":
            continue
        py_type = hints.get(name, str)
        props[name] = {"type": _PY_TYPE_TO_JSON.get(py_type, "string")}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    doc = (fn.__doc__ or fn.__name__).strip()
    description = doc.splitlines()[0][:1000]
    return {
        "name": fn.__name__,
        "description": description,
        "parameters": {"type": "object", "properties": props, "required": required},
    }


def _call_tool(tool_map, name, kwargs):
    fn = tool_map.get(name)
    if fn is None:
        return f"ERROR: unknown tool {name}"
    if VERBOSE:
        print(f"    [tool] {name}({kwargs})")
    try:
        return fn(**kwargs)
    except Exception as e:  # noqa: BLE001 — модель должна увидеть текст, а не падение
        return f"ERROR: {type(e).__name__}: {e}"


class BaseLLMClient:
    """Общий интерфейс: .prompt(message, tools=None) -> str (как kbench.llms[model_id])."""

    def __init__(self, model, system_prompt=None):
        self.model = model
        self.system_prompt = system_prompt
        self.history = []
        self.usage = {"input_tokens": 0, "output_tokens": 0, "calls": 0}

    def _record_usage(self, inp, out):
        self.usage["input_tokens"] += inp or 0
        self.usage["output_tokens"] += out or 0
        self.usage["calls"] += 1

    def prompt(self, message, tools=None):
        raise NotImplementedError


In [ ]:
class AnthropicLLMClient(BaseLLMClient):
    def __init__(self, model, system_prompt=None):
        super().__init__(model, system_prompt)
        import anthropic
        self._client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    def prompt(self, message, tools=None):
        self.history.append({"role": "user", "content": message})

        tool_defs, tool_map = None, {}
        if tools:
            tool_defs = []
            for fn in tools:
                schema = _tool_schema(fn)
                tool_defs.append({
                    "name": schema["name"],
                    "description": schema["description"],
                    "input_schema": schema["parameters"],
                })
                tool_map[schema["name"]] = fn

        for _ in range(TOOL_ROUND_LIMIT):
            kwargs = dict(model=self.model, max_tokens=4096, messages=self.history)
            if self.system_prompt:
                kwargs["system"] = self.system_prompt
            if tool_defs:
                kwargs["tools"] = tool_defs
            resp = self._client.messages.create(**kwargs)
            self._record_usage(resp.usage.input_tokens, resp.usage.output_tokens)
            self.history.append({"role": "assistant", "content": resp.content})

            tool_uses = [b for b in resp.content if b.type == "tool_use"]
            if not tool_uses:
                return "".join(b.text for b in resp.content if b.type == "text")

            results = []
            for call in tool_uses:
                out = _call_tool(tool_map, call.name, call.input or {})
                results.append({
                    "type": "tool_result", "tool_use_id": call.id, "content": str(out),
                })
            self.history.append({"role": "user", "content": results})

        raise ToolRoundLimitExceeded(
            f"ToolInvocationLimitExhausted: exceeded {TOOL_ROUND_LIMIT} tool round(s)"
        )


In [ ]:
class OpenAILLMClient(BaseLLMClient):
    def __init__(self, model, system_prompt=None):
        super().__init__(model, system_prompt)
        import openai
        self._client = openai.OpenAI(api_key=OPENAI_API_KEY)
        if system_prompt:
            self.history.append({"role": "system", "content": system_prompt})

    def prompt(self, message, tools=None):
        self.history.append({"role": "user", "content": message})

        tool_defs, tool_map = None, {}
        if tools:
            tool_defs = []
            for fn in tools:
                schema = _tool_schema(fn)
                tool_defs.append({"type": "function", "function": {
                    "name": schema["name"],
                    "description": schema["description"],
                    "parameters": schema["parameters"],
                }})
                tool_map[schema["name"]] = fn

        for _ in range(TOOL_ROUND_LIMIT):
            kwargs = dict(model=self.model, messages=self.history)
            if tool_defs:
                kwargs["tools"] = tool_defs
            resp = self._client.chat.completions.create(**kwargs)
            usage = resp.usage
            self._record_usage(
                getattr(usage, "prompt_tokens", 0), getattr(usage, "completion_tokens", 0)
            )
            msg = resp.choices[0].message
            self.history.append(msg.model_dump(exclude_none=True))

            if not msg.tool_calls:
                return msg.content or ""

            for call in msg.tool_calls:
                try:
                    args = json.loads(call.function.arguments or "{}")
                except json.JSONDecodeError as e:
                    self.history.append({
                        "role": "tool", "tool_call_id": call.id,
                        "content": f"ERROR: bad arguments JSON: {e}",
                    })
                    continue
                out = _call_tool(tool_map, call.function.name, args)
                self.history.append({
                    "role": "tool", "tool_call_id": call.id, "content": str(out),
                })

        raise ToolRoundLimitExceeded(
            f"ToolInvocationLimitExhausted: exceeded {TOOL_ROUND_LIMIT} tool round(s)"
        )


In [ ]:
class GoogleLLMClient(BaseLLMClient):
    def __init__(self, model, system_prompt=None):
        super().__init__(model, system_prompt)
        from google import genai
        from google.genai import types
        self._types = types
        self._client = genai.Client(api_key=GOOGLE_API_KEY)

    def prompt(self, message, tools=None):
        types = self._types
        self.history.append(types.Content(role="user", parts=[types.Part(text=message)]))

        gtools, tool_map = None, {}
        if tools:
            decls = []
            for fn in tools:
                schema = _tool_schema(fn)
                decls.append(types.FunctionDeclaration(
                    name=schema["name"], description=schema["description"],
                    parameters=schema["parameters"],
                ))
                tool_map[schema["name"]] = fn
            gtools = [types.Tool(function_declarations=decls)]

        config = types.GenerateContentConfig(
            system_instruction=self.system_prompt, tools=gtools,
        )

        for _ in range(TOOL_ROUND_LIMIT):
            resp = self._client.models.generate_content(
                model=self.model, contents=self.history, config=config,
            )
            um = resp.usage_metadata
            self._record_usage(
                getattr(um, "prompt_token_count", 0), getattr(um, "candidates_token_count", 0)
            )
            cand = resp.candidates[0]
            self.history.append(cand.content)

            calls = [p.function_call for p in cand.content.parts if p.function_call]
            if not calls:
                return resp.text or ""

            parts = []
            for call in calls:
                out = _call_tool(tool_map, call.name, dict(call.args or {}))
                parts.append(types.Part.from_function_response(
                    name=call.name, response={"result": str(out)}
                ))
            self.history.append(types.Content(role="user", parts=parts))

        raise ToolRoundLimitExceeded(
            f"ToolInvocationLimitExhausted: exceeded {TOOL_ROUND_LIMIT} tool round(s)"
        )


### Реестр моделей (замена `kbench.llms[model_id]` / `MODELS` из раздела 6)

Каждая запись — `label` (используется для slug ветки, как раньше `model_id`),
`provider` (`anthropic` / `openai` / `google`) и `model` — **настоящее** имя
модели у провайдера. Замените на актуальные значения перед запуском.

`PRICING` — $ за 1M токенов (input, output). Это плейсхолдеры — сверьте с
актуальными ценами провайдера перед тем, как полагаться на `cost_usd` в отчёте.

In [ ]:
CLIENT_FACTORIES = {
    "anthropic": AnthropicLLMClient,
    "openai": OpenAILLMClient,
    "google": GoogleLLMClient,
}

# TODO: проверьте актуальные названия моделей и цены перед прогоном.
MODELS = [
    {"label": "claude-opus", "provider": "anthropic", "model": "claude-opus-4-1-20250805"},
    {"label": "gemini-flash", "provider": "google", "model": "gemini-2.5-flash"},
    {"label": "gpt-mini", "provider": "openai", "model": "gpt-4.1-mini"},
]

# $ за 1M токенов: (input, output). Плейсхолдеры — обновите перед использованием.
PRICING = {
    "claude-opus-4-1-20250805": (15.0, 75.0),
    "gemini-2.5-flash": (0.30, 2.50),
    "gpt-4.1-mini": (0.40, 1.60),
}


def make_llm_client(entry, system_prompt=None):
    cls = CLIENT_FACTORIES[entry["provider"]]
    return cls(entry["model"], system_prompt=system_prompt)


def estimate_cost_usd(model_name, input_tokens, output_tokens):
    price = PRICING.get(model_name)
    if not price:
        return None
    in_price, out_price = price
    return input_tokens / 1e6 * in_price + output_tokens / 1e6 * out_price


## 5. Задача `github_issue_resolution`

Успех засчитывается **только** если одновременно:
1. ветка `<slug>-issue-<N>` существует и опережает `main` хотя бы на 1 коммит;
2. в issue есть комментарий со ссылкой на эту ветку.

`run_rounds`, `StepBudget`, журнал вызовов и его сжатие — без изменений
относительно оригинала; убраны только `kbench.task`/`kbench.chats`/
`kbench.assertions`, которые заменены на обычный python-объект результата.


In [ ]:
ISSUE_SYSTEM_PROMPT = (
    "Ты — инженер-агент, работающий с реальным GitHub-репозиторием только через "
    "инструменты. Твой цикл работы: прочитать issue -> изучить релевантные файлы "
    "(list_files/search_code/read_file) -> создать ветку под issue "
    "(create_issue_branch) -> внести изменения (write_file/patch_file) -> "
    "проверить (diff_vs_base) -> отчитаться в issue (report_work). "
    "Никогда не выдумывай содержимое файлов — сначала читай."
)

ISSUE_ROUNDS_LOG = []

# Порог сырого лога вызовов (в символах), после которого он схлопывается в
# резюме через отдельный вызов LLM, а не растёт неограниченно из раунда в раунд.
CALL_LOG_CHAR_LIMIT = 120000

SUMMARIZE_SYSTEM_PROMPT = (
    "Ты сжимаешь журнал вызовов инструментов GitHub-агента в компактный статус. "
    "На входе — предыдущее краткое резюме (может быть пустым) и новые сырые записи "
    "вызовов инструментов. Выдай ОДИН компактный абзац на русском: что уже сделано "
    "(создана ли ветка, какие файлы записаны, был ли report_work), без пересказа "
    "промежуточных read/list-вызовов, если они не дали важной информации. "
    "Не выдумывай факты, которых нет в записях."
)


def format_log_entries(entries):
    return "\n".join(
        f"- {e['tool']}({e['kwargs']}) -> {e['result_preview']}" for e in entries
    ) or "(вызовов не было)"


def summarize_call_log(llm, log_summary, entries):
    """Сжимает call_log через LLM. При сбое — безопасный фолбэк вместо падения задачи."""
    raw_text = format_log_entries(entries)
    prompt = (
        f"{SUMMARIZE_SYSTEM_PROMPT}\n\n"
        f"Предыдущее резюме:\n{log_summary or '(пусто, это первое сжатие)'}\n\n"
        f"Новые записи с прошлого сжатия:\n{raw_text}\n\n"
        "Дай новое единое резюме, учитывающее и предыдущее, и новые записи."
    )
    try:
        # без tools — это чисто текстовый вызов на суммаризацию, не агентный шаг
        summary = llm.prompt(prompt)
        return summary.strip()
    except Exception as e:  # noqa: BLE001 — сжатие не должно ронять задачу
        fallback = (
            f"{log_summary or ''}\n[сжатие через LLM не удалось "
            f"({type(e).__name__}): оставлены только имена вызванных инструментов] "
            + ", ".join(e["tool"] for e in entries)
        ).strip()
        return fallback


def run_rounds(llm, make_round_tools, task_check_fn, task_context):
    self_report = None
    call_log = []      # свежие сырые записи с прошлого сжатия (общие на все раунды)
    log_summary = ""   # накопленное сжатое резюме более ранних записей

    for round_num in range(1, MAX_ROUNDS + 1):
        budget = StepBudget(STEPS_PER_ROUND, log=call_log)
        tools = make_round_tools(budget)
        if round_num == 1:
            msg = (
                f"{task_context}\n\nУ тебя до {STEPS_PER_ROUND} вызовов инструментов "
                f"на раунд и до {MAX_ROUNDS} раундов."
            )
        else:
            raw_text = format_log_entries(call_log)
            if len(raw_text) > CALL_LOG_CHAR_LIMIT:
                log_summary = summarize_call_log(llm, log_summary, call_log)
                call_log.clear()
                raw_text = format_log_entries(call_log)

            history = (
                (f"Резюме прошлых раундов:\n{log_summary}\n\n" if log_summary else "")
                + f"Вызовы с прошлого резюме:\n{raw_text}"
            )
            msg = (
                f"Раунд {round_num}/{MAX_ROUNDS}. Новый бюджет: {STEPS_PER_ROUND} "
                f"вызовов.\n{history}\n"
                "Не повторяй уже сделанные вызовы без необходимости, продолжай "
                "со следующего логического шага."
            )
        if round_num == MAX_ROUNDS:
            msg += (
                "\n\nПОСЛЕДНИЙ РАУНД. В конце добавь блок:\n===SELF_REPORT===\n"
                "готово_процентов: <0-100>\nосталось_сделать: <кратко>\n"
                "нужно_ещё_шагов: <число>\n===END_SELF_REPORT==="
            )
        try:
            response = llm.prompt(msg, tools=tools)
        except Exception as e:  # noqa: BLE001 — например ToolRoundLimitExceeded
            self_report = f"раунд {round_num} прерван исключением: {type(e).__name__}: {e}"
            if task_check_fn():
                return True, round_num, self_report
            continue
        m = re.search(r"===SELF_REPORT===(.*?)===END_SELF_REPORT===", response, re.DOTALL)
        if m:
            self_report = m.group(1).strip()
        if task_check_fn():
            return True, round_num, self_report
    return False, MAX_ROUNDS, self_report


def check_issue_solved(slug, issue_number):
    branch = f"{slug}-issue-{issue_number}"
    cmp = gh_compare(GITHUB_BASE_BRANCH, branch)
    if not cmp or cmp["ahead_by"] < 1:
        return False
    try:
        comments = _gh_paginate(f"{REPO_PATH}/issues/{issue_number}/comments")
    except RuntimeError:
        return False
    return any(branch in (c.get("body") or "") for c in comments)


def github_issue_resolution_eval(entry, issue_number):
    """Прогоняет задачу для одной модели (entry из MODELS) на одном issue.

    Возвращает dict-результат — раньше эту роль играл @kbench.task +
    .evaluate(); assert_fail заменён на явное поле "solved"/"error" вместо
    исключения, чтобы раннер (секция 6) мог продолжать по остальным issues.
    """
    slug = slugify_model_id(entry["label"])
    issue_number = int(issue_number)
    branch = f"{slug}-issue-{issue_number}"

    task_context = (
        f"Репозиторий: {GITHUB_OWNER}/{GITHUB_REPO} (база: {GITHUB_BASE_BRANCH}).\n"
        f"Задача: разбери issue #{issue_number}.\n"
        "1. get_issue — прочитай задачу целиком, включая комментарии.\n"
        "2. list_files / search_code / read_file — изучи затронутый код.\n"
        f"3. create_issue_branch({issue_number}) — создай ветку {branch}.\n"
        "4. write_file / patch_file — внеси изменения ТОЛЬКО в эту ветку.\n"
        "5. diff_vs_base — убедись, что изменения на месте.\n"
        f"6. report_work({issue_number}, '<что сделано>') — оставь комментарий "
        "в issue со ссылкой на ветку."
    )

    llm = make_llm_client(entry, system_prompt=ISSUE_SYSTEM_PROMPT)
    error = None
    try:
        solved, rounds_used, self_report = run_rounds(
            llm,
            lambda b: make_github_tools(slug, b),
            lambda: check_issue_solved(slug, issue_number),
            task_context,
        )
    except Exception as e:  # noqa: BLE001 — сбой уровня клиента (auth/сеть), не роняем раннер
        solved, rounds_used, self_report = False, 0, None
        error = f"{type(e).__name__}: {e}"

    result = {
        "model": entry["label"],
        "model_name": entry["model"],
        "provider": entry["provider"],
        "issue": issue_number,
        "branch": branch,
        "solved": solved,
        "rounds_used": rounds_used,
        "self_report": self_report,
        "error": error,
        "input_tokens": llm.usage["input_tokens"],
        "output_tokens": llm.usage["output_tokens"],
        "llm_calls": llm.usage["calls"],
        "cost_usd": estimate_cost_usd(
            entry["model"], llm.usage["input_tokens"], llm.usage["output_tokens"]
        ),
    }
    if not solved:
        result["failure_reason"] = (
            error or f"issue #{issue_number}: нет ветки {branch} с коммитами "
                     "и/или комментария со ссылкой на неё"
        )
    ISSUE_ROUNDS_LOG.append(result)
    return result


## 6. Прогон: каждая модель × каждый открытый issue

In [ ]:
open_issues = [
    it["number"]
    for it in _gh_paginate(f"{REPO_PATH}/issues", params={"state": "open"})
    if "pull_request" not in it
]
print("Открытые issues:", open_issues)

completed = []
for entry in MODELS:
    for num in open_issues:
        res = github_issue_resolution_eval(entry, num)
        completed.append(res)
        print(
            f"{entry['label']} / issue #{num}: solved={res['solved']} "
            f"rounds={res['rounds_used']} in={res['input_tokens']} "
            f"out={res['output_tokens']} cost=${res['cost_usd']}"
        )

results_df = pd.DataFrame(completed)
results_df


## 7. Дополнения — сохранение результатов и сводка по стоимости

In [ ]:
# Раньше эту роль играл archive_run_json (архивация *.run.json от kbench).
# Здесь результаты уже собраны в памяти (completed/results_df) — просто
# сохраняем их на диск, без парсинга служебных файлов фреймворка.
import json as _json
from datetime import datetime

RESULTS_PATH = f"github_benchmark_results_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    _json.dump(completed, f, ensure_ascii=False, indent=2)
print(f"Результаты сохранены: {RESULTS_PATH}")


In [ ]:
# Сводка по модели: доля решённых issue, раунды, токены, стоимость.
# Раньше это строилось из nanodollar-полей в *.run.json (load_github_benchmark_costs);
# теперь то же самое доступно напрямую из results_df.
summary = (
    results_df
    .groupby("model")
    .agg(
        solved_rate=("solved", "mean"),
        avg_rounds=("rounds_used", "mean"),
        total_input_tokens=("input_tokens", "sum"),
        total_output_tokens=("output_tokens", "sum"),
        total_cost_usd=("cost_usd", "sum"),
    )
    .reset_index()
)
summary


## Смоук-тест инструментов без LLM

Дешёвая проверка, что расширенный слой и scope токена в порядке — до того, как тратить деньги на модели.

In [ ]:
def smoke_test(slug="anthropic-claude-opus-5-default"):
    checks = {
        "branches": gh_list_branches(),
        "files(main)": gh_list_files(GITHUB_BASE_BRANCH)[:300],
        "issues(all)": gh_list_issues("all")[:500],
        "compare": gh_compare(GITHUB_BASE_BRANCH, slug),
    }
    for k, v in checks.items():
        ok = not (isinstance(v, str) and v.startswith("ERROR:"))
        print(f"[{'OK ' if ok else 'FAIL'}] {k}: {str(v)[:250]}\n")
    print("Если issues -> FAIL 403, у токена нет scope 'Issues: Read and write'.")


smoke_test()